<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 자기지도 학습 · 멀티모달 · 08. BLIP: 이해·생성과 CapFilt

## BLIP: Bootstrapping Language-Image Pre-training for Unified Vision-Language Understanding and Generation

- **원 논문:** [BLIP: Bootstrapping Language-Image Pre-training for Unified Vision-Language Understanding and Generation](https://arxiv.org/abs/2201.12086)
- **저자 / 발표:** Junnan Li, Dongxu Li, Caiming Xiong, Steven Hoi · ICML (2022)
- **난이도 / 예상 시간:** 고급 · 약 85분
- **선수 지식:** CLIP, image-text matching, language modeling
- **로컬 학습 데이터:** `data/field_curriculum/multimodal_pairs.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** MED의 ITC·ITM·LM 세 목적과 CapFilt의 생성/필터 단계를 작은 paired corpus에서 재현합니다.

**축소하거나 생략한 부분:** 1.29억 web pairs·ViT/BERT 대신 240개 pair와 작은 shared text embedding을 사용합니다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3.1, Figure 2 | ITC·ITM·LM 세 동작 모드 | 세 loss 유한값 |
| §3.1 Image-Text Contrastive | unimodal alignment | 대칭 CE |
| §3.1 Image-Text Matching/LM | negative match·causal token prediction | binary/next-token loss |
| §3.2, Figure 3 | captioner 생성+filter 제거 CapFilt | 정제 전후 noise |

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** image/text [B,*] → filtered pairs [B',*] → three objective scalars
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from copy import deepcopy
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

random.seed(17)
np.random.seed(17)
torch.manual_seed(17)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(17)

DATA_PATH = Path("data/field_curriculum/multimodal_pairs.npz")
assert DATA_PATH.exists(), f"먼저 로컬 학습 데이터를 준비하세요: {DATA_PATH}"
raw = np.load(DATA_PATH, allow_pickle=False)
images = torch.tensor(raw["images"], dtype=torch.float32)
tokens = torch.tensor(raw["tokens"], dtype=torch.long)
class_ids = torch.tensor(raw["class_ids"], dtype=torch.long)
sequences = torch.tensor(raw["sequences"], dtype=torch.float32)
train_idx = torch.tensor(raw["train_idx"], dtype=torch.long)
test_idx = torch.tensor(raw["test_idx"], dtype=torch.long)
assert len(images) == len(tokens) == len(class_ids) == len(sequences)
assert images.shape[1:] == (1, 8, 8) and tokens.ndim == 2 and sequences.ndim == 3
print(
    f"pairs={len(images)}, classes={class_ids.unique().numel()}, "
    f"image={tuple(images.shape[1:])}, tokens={tokens.shape[1]}"
)
print(ACCELERATOR.summary())
print("dataset/index preprocessing stays on CPU, while neural training uses DEVICE")

## 포트폴리오 해설: 표현, 목적함수, 업데이트

$$\mathcal L_{BLIP}=\mathcal L_{ITC}+\mathcal L_{ITM}+\mathcal L_{LM}$$

- **기호 정의:** ITC는 정렬, ITM은 pair 일치, LM은 다음 token 생성 목적입니다.
- **수식의 역할:** 하나의 모델에 이해와 생성을 결합하고 생성·필터 과정으로 caption을 정제합니다.
- **구현 이유:** representation 붕괴·modality 누설·잘못된 positive 연결을
  코드에서 확인할 수 있도록 핵심 수식을 view 준비와 loss 경계로 분리했습니다.
- **Task/코드 대응:** Task P의 `prepare_views`, 논문 전용 `nn.Module`의
  `forward`, `compute_loss`, `training_step`이 §3.1, Figure 2을 구현합니다.
- **입출력 shape:** image [B, 1, 8, 8], token [B, L] → contrastive [B, B], LM [B, L-1, V]
- **평가:** 세 loss, retrieval, clean/noisy pair filtering precision
- **원문 대비 한계:** 1.29억 web pairs·ViT/BERT 대신 240개 pair와 작은 shared text embedding을 사용합니다.

구현은 `로컬 데이터 통계 → view/modality 준비 → model → objective/update → 평가`
순서입니다. 이 순서는 데이터 누설과 stop-gradient 경계를 코드에서 찾기 쉽게 합니다.

### 구현 단계 2 · 합성 입력·데이터 계약 (data-contract 예외)

- **원문 위치:** 해당 없음 — 원 논문의 학습 연산이 아니라 외부 다운로드를 없애기 위한 축소 입력 생성 단계입니다.
- **Tensor shape 계약:** image/text [B,*] → filtered pairs [B',*] → three objective scalars
- **구현 이유:** 논문 메커니즘과 데이터 규모 효과를 분리하고 Windows CPU에서도 같은 입력을 재현합니다.
- **완료 증거:** pair 정렬, batch 크기, dtype과 입력 rank assertion이 통과해야 합니다.

In [ ]:
image_features = torch.tensor(raw["image_features"], dtype=torch.float32)
class_image_features = torch.tensor(raw["class_image_features"], dtype=torch.float32)
class_tokens = torch.stack(
    [tokens[torch.where(class_ids == c)[0][0]] for c in class_ids.unique(sorted=True)]
)

### 구현 단계 3 · 합성 입력·데이터 계약 (data-contract 예외)

- **원문 위치:** 해당 없음 — 원 논문의 학습 연산이 아니라 외부 다운로드를 없애기 위한 축소 입력 생성 단계입니다.
- **Tensor shape 계약:** image/text [B,*] → filtered pairs [B',*] → three objective scalars
- **구현 이유:** 논문 메커니즘과 데이터 규모 효과를 분리하고 Windows CPU에서도 같은 입력을 재현합니다.
- **완료 증거:** pair 정렬, batch 크기, dtype과 입력 rank assertion이 통과해야 합니다.

In [ ]:
def make_web_and_synthetic(text, image_feat, rate=0.35):
    web = text.clone()
    count = int(len(text) * rate)
    bad = torch.arange(count)
    web[bad] = web[bad.roll(1)]
    nearest = (
        F.normalize(image_feat, dim=1) @ F.normalize(class_image_features, dim=1).T
    ).argmax(1)
    synthetic = class_tokens[nearest]
    return web, synthetic, bad


web, synthetic, bad = make_web_and_synthetic(tokens, image_features)
assert len(bad) == int(len(tokens) * 0.35) and synthetic.shape == tokens.shape

### 구현 단계 4 · 핵심 연산·모델

- **원문 위치:** §3.2, Figure 3
- **이 셀의 책임:** captioner 생성+filter 제거 CapFilt
- **Tensor shape 계약:** image/text [B,*] → filtered pairs [B',*] → three objective scalars
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 정제 전후 noise. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def capfilt(web: Tensor, synthetic: Tensor):
    keep = (web == synthetic).float().mean(1) >= 0.5
    cleaned = web.clone()
    cleaned[~keep] = synthetic[~keep]
    return cleaned, keep


cleaned, keep = capfilt(web, synthetic)
before = float((web != tokens).any(1).float().mean())
after = float((cleaned != tokens).any(1).float().mean())
assert keep.dtype == torch.bool and after < before

### 구현 단계 5 · 핵심 연산·모델

- **원문 위치:** §3.1, Figure 2 / §3.1 Image-Text Contrastive / §3.1 Image-Text Matching/LM
- **이 셀의 책임:** ITC·ITM·LM 세 동작 모드 / unimodal alignment / negative match·causal token prediction
- **Tensor shape 계약:** image/text [B,*] → filtered pairs [B',*] → three objective scalars
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 세 loss 유한값 / 대칭 CE / binary/next-token loss. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
class TinyBLIP(nn.Module):
    def __init__(self, vocab, dim=16):
        super().__init__()
        self.vocab = vocab
        self.image = nn.Sequential(nn.Flatten(), nn.Linear(64, dim))
        self.word = nn.Embedding(vocab, dim)
        self.itm = nn.Linear(2 * dim, 1)
        self.lm = nn.Linear(2 * dim, vocab)

    def losses(self, x, t):
        zi = F.normalize(self.image(x), dim=1)
        zt = F.normalize(self.word(t).mean(1), dim=1)
        target = torch.arange(len(x), device=x.device)
        sim = zi @ zt.T / 0.15
        itc = (F.cross_entropy(sim, target) + F.cross_entropy(sim.T, target)) / 2
        neg = zt.roll(1, 0)
        match_logits = torch.cat(
            [self.itm(torch.cat([zi, zt], 1)), self.itm(torch.cat([zi, neg], 1))]
        )
        itm_target = torch.cat(
            [
                torch.ones(len(x), 1, device=x.device),
                torch.zeros(len(x), 1, device=x.device),
            ]
        )
        itm = F.binary_cross_entropy_with_logits(match_logits, itm_target)
        context = zi[:, None, :].expand(-1, t.shape[1] - 1, -1)
        lm_logits = self.lm(torch.cat([self.word(t[:, :-1]), context], 2))
        lm = F.cross_entropy(lm_logits.reshape(-1, self.vocab), t[:, 1:].reshape(-1))
        return itc, itm, lm, sim


blip = TinyBLIP(int(tokens.max()) + 1)
parts = blip.losses(images[:24], cleaned[:24])
assert all(torch.isfinite(v) for v in parts[:3])

### 구현 단계 6 · 평가·완료 증거

- **원문 위치:** §3.1, Figure 2 / §3.1 Image-Text Contrastive / §3.1 Image-Text Matching/LM / §3.2, Figure 3
- **이 셀의 책임:** ITC·ITM·LM 세 동작 모드 / unimodal alignment / negative match·causal token prediction / captioner 생성+filter 제거 CapFilt
- **Tensor shape 계약:** image/text [B,*] → filtered pairs [B',*] → three objective scalars
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 세 loss 유한값 / 대칭 CE / binary/next-token loss / 정제 전후 noise. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
blip = ACCELERATOR.move(blip)
idx = train_idx[:96]
train_images, train_cleaned = ACCELERATOR.move(images[idx], cleaned[idx])
optimizer = torch.optim.Adam(blip.parameters(), lr=0.012)
losses = []
for _ in range(130):
    itc, itm, lm, _ = blip.losses(train_images, train_cleaned)
    loss = itc + itm + 0.5 * lm
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(float(loss.detach().cpu()))
with torch.no_grad():
    test_images, test_cleaned = ACCELERATOR.move(images[test_idx], cleaned[test_idx])
    *_, sim = blip.losses(test_images, test_cleaned)
    retrieval = float(
        (sim.argmax(1) == torch.arange(len(test_idx), device=sim.device)).float().mean()
    )
assert losses[-1] < losses[0] and after < before
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].plot(losses)
ax[0].set_title("ITC+ITM+LM")
ax[1].bar(["web", "CapFilt"], [before, after])
ax[1].set_title("caption noise rate")
plt.tight_layout()
plt.show()
print(
    {"noise_before": before, "noise_after": after, "retrieval@1": round(retrieval, 3)}
)

### Task P1 · modality/view 준비와 핵심 모델

image·text·sequence의 shape를 먼저 고정하고 augmentation 또는 modality
경계를 명시한 뒤 논문 전용 `nn.Module`을 구성합니다.

### 구현 단계 7 · 핵심 연산·모델

- **원문 위치:** §3.1, Figure 2 / §3.1 Image-Text Contrastive / §3.1 Image-Text Matching/LM
- **이 셀의 책임:** ITC·ITM·LM 세 동작 모드 / unimodal alignment / negative match·causal token prediction
- **Tensor shape 계약:** image/text [B,*] → filtered pairs [B',*] → three objective scalars
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 세 loss 유한값 / 대칭 CE / binary/next-token loss. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    hidden_dim: int = 16
    learning_rate: float = 0.02
    steps: int = 14


def prepare_views(image_batch: Tensor, text_batch: Tensor) -> tuple[Tensor, Tensor]:
    return image_batch, text_batch


class BlipPortfolioModel(nn.Module):
    def __init__(self, config: PortfolioConfig):
        super().__init__()
        self.config = config
        self.vocabulary = int(tokens.max()) + 1
        self.image_encoder = nn.Sequential(
            nn.Flatten(), nn.Linear(64, config.hidden_dim)
        )
        self.token_embedding = nn.Embedding(self.vocabulary, config.hidden_dim)
        self.matching_head = nn.Linear(config.hidden_dim * 2, 1)
        self.language_model = nn.GRU(
            config.hidden_dim, config.hidden_dim, batch_first=True
        )
        self.token_head = nn.Linear(config.hidden_dim, self.vocabulary)

    def forward(self, image_batch: Tensor, text_batch: Tensor) -> dict[str, Tensor]:
        image_embedding = self.image_encoder(image_batch)
        token_hidden = self.token_embedding(text_batch)
        text_embedding = token_hidden.mean(dim=1)
        contrastive = F.normalize(image_embedding, dim=1)
        contrastive = contrastive @ F.normalize(text_embedding, dim=1).T
        positive_pair = torch.cat([image_embedding, text_embedding], dim=1)
        negative_pair = torch.cat(
            [image_embedding, text_embedding.roll(shifts=1, dims=0)], dim=1
        )
        matching = self.matching_head(torch.cat([positive_pair, negative_pair], dim=0))
        # BLIP §3.1 LM mode: image representation이 decoder initial state를
        # condition하므로 caption token 예측이 image에 따라 달라집니다.
        lm_hidden, _ = self.language_model(
            token_hidden[:, :-1], image_embedding.unsqueeze(0)
        )
        token_logits = self.token_head(lm_hidden)
        return {"contrastive": contrastive, "matching": matching, "lm": token_logits}

    def compute_loss(self, outputs: dict[str, Tensor], text_batch: Tensor) -> Tensor:
        target = torch.arange(len(text_batch), device=text_batch.device)
        contrastive = 0.5 * (
            F.cross_entropy(outputs["contrastive"], target)
            + F.cross_entropy(outputs["contrastive"].T, target)
        )
        matching = F.binary_cross_entropy_with_logits(
            outputs["matching"],
            torch.cat(
                [
                    torch.ones(len(text_batch), 1, device=text_batch.device),
                    torch.zeros(len(text_batch), 1, device=text_batch.device),
                ],
                dim=0,
            ),
        )
        language = F.cross_entropy(
            outputs["lm"].reshape(-1, self.vocabulary),
            text_batch[:, 1:].reshape(-1),
        )
        return contrastive + matching + language

    def training_step(self, image_batch: Tensor, text_batch: Tensor) -> Tensor:
        return self.compute_loss(self(image_batch, text_batch), text_batch)

### Task P2 · objective, stop-gradient, update

positive/negative, online/target 또는 masked/visible 경계를 확인하면서
`fit_portfolio_model`의 gradient와 EMA update 순서를 추적합니다.

### 구현 단계 8 · 목적함수·학습 update

- **원문 위치:** §3.1, Figure 2
- **이 셀의 책임:** ITC·ITM·LM 세 동작 모드
- **Tensor shape 계약:** image/text [B,*] → filtered pairs [B',*] → three objective scalars
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 세 loss 유한값. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(
    model: BlipPortfolioModel,
    image_batch: Tensor,
    text_batch: Tensor,
) -> list[float]:
    optimizer = torch.optim.Adam(model.parameters(), lr=model.config.learning_rate)
    history = []
    for _ in range(model.config.steps):
        loss = model.training_step(image_batch, text_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(float(loss.detach().cpu()))
    return history

### Task P3 · 표현 품질 평가

학습 loss와 retrieval·reconstruction·conditioning 지표를 분리해 계산합니다.
작은 로컬 pair의 수치는 원 논문의 downstream benchmark와 직접 비교하지 않습니다.

### 구현 단계 9 · 평가·완료 증거

- **원문 위치:** §3.1 Image-Text Contrastive / §3.1 Image-Text Matching/LM / §3.2, Figure 3
- **이 셀의 책임:** unimodal alignment / negative match·causal token prediction / captioner 생성+filter 제거 CapFilt
- **Tensor shape 계약:** image/text [B,*] → filtered pairs [B',*] → three objective scalars
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 대칭 CE / binary/next-token loss / 정제 전후 noise. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def evaluate_portfolio_model(
    model: BlipPortfolioModel,
    image_batch: Tensor,
    text_batch: Tensor,
    keep_mask: Tensor,
) -> dict[str, float]:
    with torch.no_grad():
        outputs = model(image_batch, text_batch)
        similarity = outputs["contrastive"]
        target = torch.arange(len(similarity), device=similarity.device)
        recall = (similarity.argmax(1) == target).float().mean()
        rolled_outputs = model(image_batch.roll(shifts=1, dims=0), text_batch)
        conditioning_gap = (outputs["lm"] - rolled_outputs["lm"]).abs().mean()
        positive_logit, negative_logit = outputs["matching"].chunk(2, dim=0)
        matching_margin = positive_logit.mean() - negative_logit.mean()
        retained_count = int(keep_mask.sum().item())
        replaced_count = int((~keep_mask).sum().item())
    return {
        "retrieval_recall1": float(recall.cpu()),
        "lm_image_conditioning_gap": float(conditioning_gap.cpu()),
        "itm_positive_negative_margin": float(matching_margin.cpu()),
        "capfilt_retained_count": float(retained_count),
        "capfilt_replaced_count": float(replaced_count),
    }


portfolio_train_index = train_idx[:96]
portfolio_test_index = test_idx
assert not torch.isin(portfolio_train_index, portfolio_test_index).any()
portfolio_train_images, portfolio_cleaned_tokens = prepare_views(
    images[portfolio_train_index],
    cleaned[portfolio_train_index],
)
portfolio_train_images, portfolio_cleaned_tokens = ACCELERATOR.move(
    portfolio_train_images,
    portfolio_cleaned_tokens,
)
(
    portfolio_test_images,
    portfolio_test_cleaned_tokens,
    portfolio_test_keep,
) = ACCELERATOR.move(
    images[portfolio_test_index],
    cleaned[portfolio_test_index],
    keep[portfolio_test_index],
)
portfolio_model = ACCELERATOR.move(BlipPortfolioModel(PortfolioConfig()))
portfolio_history = fit_portfolio_model(
    portfolio_model,
    portfolio_train_images,
    portfolio_cleaned_tokens,
)
portfolio_metrics = evaluate_portfolio_model(
    portfolio_model,
    portfolio_test_images,
    portfolio_test_cleaned_tokens,
    portfolio_test_keep,
)
assert np.isfinite(portfolio_history).all()
assert portfolio_metrics["lm_image_conditioning_gap"] > 0.0
assert portfolio_metrics["capfilt_retained_count"] + portfolio_metrics[
    "capfilt_replaced_count"
] == len(portfolio_test_index)
print(portfolio_metrics)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 1.29억 web pairs·ViT/BERT 대신 240개 pair와 작은 shared text embedding을 사용합니다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.